In [1]:
import kaleidocell
import scanpy as sc
import os


In [2]:
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_subsets/Mast.h5ad")


In [3]:
print(adata.obs.columns.tolist())
print(adata.obs["donor_id"].nunique())
print(f"{adata.obs['annotation_level_3'].unique()[0]}: {adata.n_obs} cells")

['author', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'annotation_level_1', 'annotation_level_2', 'annotation_level_3', 'gbmap', 'suspension_type', 'stage', 'location', 'sector', 'celltype_original', 'EGFR', 'MET', 'p53', 'TERT', 'ATRX', 'PTEN', 'MGMT', 'chr1p19q', 'PDGFR', 'tissue_ontology_term_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid']
78
Mast: 816 cells


In [4]:
results_nmf, nmf_convergence = kaleidocell.multi_sample_nmf(
    adata,
    batch_key="donor_id",
    test_ranks=[4, 5, 6, 7, 8, 9],
    n_initializations=1,
    max_iterations=100
)

Running multi-sample NMF on 78 samples.
Tested ranks: [4, 5, 6, 7, 8, 9]


Global NMF Progress:   0%|          | 0/46800 [00:00<?, ?it/s]

/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/pr

Multi-sample NMF complete.


In [5]:
results_mp = kaleidocell.derive_nmf_metaprograms(results_nmf)
print(results_mp["metrics"])

Total number of programs: 3042
Optimal number of meta-programs: 30
Clustering using hierarchical Ward linkage
Defining meta-program consensus
Computing meta-program metrics
Meta-program derivation complete.
      sampleCoverage  silhouette  meanSimilarity  nPrograms  nGenes
MP1         0.012821    0.399276           0.426         18     647
MP2         0.038462    0.079280           0.133         61    1499
MP3         0.500000   -0.013458           0.039       1192    5901
MP4         0.025641    0.253280           0.317         28     489
MP5         0.012821    0.298953           0.372         19     369
MP6         0.051282    0.049991           0.110         72     564
MP7         0.012821    0.377349           0.432         19     563
MP8         0.012821    0.383801           0.433         18     554
MP9         0.243590    0.016788           0.109        246    2614
MP10        0.192308    0.700566           0.772         49       1
MP11        0.192308    0.275194           0.

In [6]:
mp_scores = kaleidocell.compute_mp_scores(results_mp, adata)

Computing module scores per MP.
Finished computing MP scores.


In [7]:
path = kaleidocell.get_html(
    results_mp,
    adata,
    mp_scores=mp_scores,
    obs=["annotation_level_3", "stage", "donor_id", "disease"],
    output_path="/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Mast"
)
print(f"Report written to: {path}")

Rendering heatmap…
Rendering UMAP scores…
Building metrics table…
Running GSEA — GO Biological Process…
Ensembl gene IDs detected — translating to HGNC using bundled table.
Building gene table…
Rendering violin plots for 'annotation_level_3'…
Plotting MP distributions across 'annotation_level_3'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'stage'…
Plotting MP distributions across 'stage'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'donor_id'…
Plotting MP distributions across 'donor_id'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'disease'…
Plotting MP distributions across 'disease'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.

Files written to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Mast/
  results.html
  gsea_GO_Biological_Process.csv
  genes.csv
Report written to: /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Mast/results.html
